# 修正索引名稱為 datetime

這個筆記本用於將所有 parquet 檔案的索引名稱從 `k_time` 改為 `datetime`。

In [ ]:
import pandas as pd
import os
import glob
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

## 1. 檢查當前狀態

In [ ]:
# 設定資料目錄
data_dir = r'D:\03_預估量相關資量\tw_kbar_1m_vol_exp'

# 取得所有檔案
pattern = os.path.join(data_dir, 'vol_exp_*.parquet')
all_files = glob.glob(pattern)
all_files = [f for f in all_files if '_backup' not in f]

print(f"資料目錄: {data_dir}")
print(f"找到 {len(all_files)} 個 parquet 檔案\n")

In [ ]:
# 檢查前5個檔案的索引狀態
def check_files_status(files, limit=5):
    """檢查檔案的索引狀態"""
    results = []
    
    for file_path in files[:limit]:
        filename = os.path.basename(file_path)
        try:
            df = pd.read_parquet(file_path)
            results.append({
                '檔案': filename,
                '索引名稱': df.index.name,
                '索引類型': str(df.index.dtype),
                '第一個時間': str(df.index[0]) if len(df.index) > 0 else 'N/A',
                '資料形狀': f"{df.shape[0]} x {df.shape[1]}",
                '需要修正': '是' if df.index.name != 'datetime' else '否'
            })
        except Exception as e:
            results.append({
                '檔案': filename,
                '索引名稱': 'Error',
                '索引類型': str(e),
                '第一個時間': 'N/A',
                '資料形狀': 'N/A',
                '需要修正': '錯誤'
            })
    
    df_results = pd.DataFrame(results)
    return df_results

# 顯示當前狀態
print("當前索引狀態（前5個檔案）：")
df_status = check_files_status(all_files, limit=5)
display(df_status)

## 2. 單一檔案修正測試

In [ ]:
def fix_single_file(file_path, save=False):
    """修正單一檔案的索引名稱"""
    filename = os.path.basename(file_path)
    print(f"處理檔案: {filename}")
    
    # 讀取檔案
    df = pd.read_parquet(file_path)
    
    print(f"\n修正前：")
    print(f"  索引名稱: {df.index.name}")
    print(f"  索引類型: {df.index.dtype}")
    print(f"  索引範例: {df.index[:3].tolist()}")
    
    # 修正索引名稱
    if df.index.name != 'datetime':
        df.index.name = 'datetime'
        
        print(f"\n修正後：")
        print(f"  索引名稱: {df.index.name}")
        print(f"  索引類型: {df.index.dtype}")
        
        if save:
            # 備份原始檔案
            backup_path = file_path.replace('.parquet', '_name_backup.parquet')
            if not os.path.exists(backup_path):
                # 讀取原始檔案並備份
                df_original = pd.read_parquet(file_path)
                df_original.to_parquet(backup_path)
                print(f"\n✓ 已備份至: {os.path.basename(backup_path)}")
            
            # 儲存修正後的檔案
            df.to_parquet(file_path)
            print(f"✓ 已儲存修正後的檔案")
    else:
        print(f"\n索引名稱已經是 'datetime'，無需修正")
    
    return df

# 測試修正（不儲存）
if all_files:
    print("測試修正第一個檔案（不儲存）：\n")
    df_test = fix_single_file(all_files[0], save=False)
    
    print("\n修正後的資料預覽：")
    display(df_test.head(3))

## 3. 批次修正索引名稱

In [ ]:
def batch_fix_index_names(files, test_mode=True, backup=True):
    """批次修正所有檔案的索引名稱"""
    
    if test_mode:
        files = files[:3]
        print(f"測試模式：只處理前 {len(files)} 個檔案\n")
    else:
        print(f"即將處理 {len(files)} 個檔案\n")
    
    results = []
    success_count = 0
    skip_count = 0
    error_count = 0
    
    for i, file_path in enumerate(files, 1):
        filename = os.path.basename(file_path)
        print(f"[{i}/{len(files)}] {filename}")
        
        try:
            # 讀取檔案
            df = pd.read_parquet(file_path)
            original_name = df.index.name
            
            if original_name != 'datetime':
                # 備份（如果需要）
                if backup:
                    backup_path = file_path.replace('.parquet', '_name_backup.parquet')
                    if not os.path.exists(backup_path):
                        df_backup = pd.read_parquet(file_path)
                        df_backup.to_parquet(backup_path)
                
                # 修正索引名稱
                df.index.name = 'datetime'
                
                # 儲存
                df.to_parquet(file_path)
                
                print(f"  ✓ 已修正: '{original_name}' -> 'datetime'")
                success_count += 1
                results.append({'檔案': filename, '狀態': '✓ 已修正', '原索引': original_name, '新索引': 'datetime'})
            else:
                print(f"  ○ 已是正確名稱，跳過")
                skip_count += 1
                results.append({'檔案': filename, '狀態': '○ 跳過', '原索引': 'datetime', '新索引': 'datetime'})
                
        except Exception as e:
            print(f"  ✗ 錯誤: {str(e)}")
            error_count += 1
            results.append({'檔案': filename, '狀態': '✗ 錯誤', '原索引': 'N/A', '新索引': str(e)})
    
    print(f"\n{'='*60}")
    print(f"處理完成：")
    print(f"  ✓ 修正: {success_count}")
    print(f"  ○ 跳過: {skip_count}")
    print(f"  ✗ 錯誤: {error_count}")
    print(f"  總計: {len(files)}")
    
    return pd.DataFrame(results)

# 執行測試批次（只處理3個檔案）
print("執行測試批次修正：")
df_test_results = batch_fix_index_names(all_files, test_mode=True, backup=True)
display(df_test_results)

## 4. 驗證修正結果

In [ ]:
# 重新檢查檔案狀態
print("修正後的狀態（前5個檔案）：")
df_status_after = check_files_status(all_files, limit=5)
display(df_status_after)

## 5. 處理所有檔案（生產環境）

In [ ]:
# ⚠️ 注意：這會處理所有檔案！
# 請確認測試沒問題後再執行

def fix_all_files():
    """修正所有檔案的索引名稱"""
    
    # 確認執行
    user_input = input("確定要修正所有檔案的索引名稱嗎？輸入 'YES' 繼續: ")
    if user_input != 'YES':
        print("已取消")
        return None
    
    # 執行批次修正
    df_results = batch_fix_index_names(all_files, test_mode=False, backup=True)
    
    # 儲存處理結果
    from datetime import datetime
    result_file = os.path.join(data_dir, f'index_fix_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv')
    df_results.to_csv(result_file, index=False, encoding='utf-8-sig')
    print(f"\n處理結果已儲存至: {result_file}")
    
    return df_results

# 執行全部修正（需要手動確認）
# df_all_results = fix_all_files()

## 6. 備份管理

In [ ]:
# 列出所有備份檔案
def list_backup_files():
    """列出所有索引名稱備份檔案"""
    pattern = os.path.join(data_dir, '*_name_backup.parquet')
    backup_files = glob.glob(pattern)
    
    print(f"找到 {len(backup_files)} 個索引名稱備份檔案")
    
    if backup_files:
        print("\n備份檔案列表：")
        total_size_mb = 0
        for f in backup_files[:10]:
            size_mb = os.path.getsize(f) / (1024 * 1024)
            total_size_mb += size_mb
            print(f"  - {os.path.basename(f)} ({size_mb:.2f} MB)")
        
        if len(backup_files) > 10:
            print(f"  ... 還有 {len(backup_files) - 10} 個檔案")
        
        # 計算總大小
        if len(backup_files) > 10:
            for f in backup_files[10:]:
                total_size_mb += os.path.getsize(f) / (1024 * 1024)
        
        print(f"\n總備份大小: {total_size_mb:.2f} MB")
    
    return backup_files

backup_list = list_backup_files()

In [ ]:
# 清理備份檔案
def clean_backup_files(confirm=False):
    """清理索引名稱備份檔案"""
    
    if not confirm:
        print("⚠️ 警告：這將永久刪除所有 *_name_backup.parquet 檔案！")
        print("請先確認修正成功後再執行")
        print("如果要執行，請設定 confirm=True")
        return
    
    pattern = os.path.join(data_dir, '*_name_backup.parquet')
    backup_files = glob.glob(pattern)
    
    deleted_count = 0
    total_size_mb = 0
    
    for backup_path in backup_files:
        try:
            size_mb = os.path.getsize(backup_path) / (1024 * 1024)
            total_size_mb += size_mb
            os.remove(backup_path)
            deleted_count += 1
        except Exception as e:
            print(f"✗ 刪除失敗 {os.path.basename(backup_path)}: {e}")
    
    print(f"\n清理完成：")
    print(f"  已刪除 {deleted_count} 個檔案")
    print(f"  釋放空間 {total_size_mb:.2f} MB")

# 清理備份（需要確認）
# clean_backup_files(confirm=False)  # 設定 confirm=True 來執行

## 7. 最終驗證

In [ ]:
# 統計所有檔案的索引名稱
def final_verification():
    """最終驗證所有檔案的索引名稱"""
    
    correct_count = 0
    incorrect_count = 0
    error_count = 0
    
    for file_path in all_files:
        try:
            df = pd.read_parquet(file_path)
            if df.index.name == 'datetime':
                correct_count += 1
            else:
                incorrect_count += 1
                print(f"需要修正: {os.path.basename(file_path)} - 索引名稱為 '{df.index.name}'")
        except Exception as e:
            error_count += 1
            print(f"讀取錯誤: {os.path.basename(file_path)} - {str(e)}")
    
    print(f"\n最終統計：")
    print(f"  ✓ 正確 (datetime): {correct_count}")
    print(f"  ✗ 需要修正: {incorrect_count}")
    print(f"  ⚠ 錯誤: {error_count}")
    print(f"  總計: {len(all_files)}")
    
    if incorrect_count == 0 and error_count == 0:
        print("\n🎉 所有檔案的索引名稱都已正確設定為 'datetime'！")
    
    return {
        'correct': correct_count,
        'incorrect': incorrect_count,
        'error': error_count
    }

# 執行最終驗證
verification_result = final_verification()